In [18]:
import pandas as pd
import re
import os

In [19]:
# load raw excel file
df_raw = pd.read_excel('../data/raw/export_search-cartel_2025-07-16_18-11.xlsx')
df_raw

,Case number,Case title,Antitrust / Cartels,Companies,Last decision date,Economic activities,Legal basis,Initiation of proceedings,Decisions,Other case related information
0,AT.40636,SNBB,Cartel,C.H. Boehringer Sohn AG & Co. KG\nhttps://comp...,04.07.2025,C.21.20 - Manufacture of pharmaceutical prepar...,Art. 101 TFEU + Art. 53 EEA,NaN,04.07.2025 - Prohibition Decision\n Decision ...,Response(s) to the Statement of Objections of ...
1,AT.40669,End-of-life vehicle recycling,Cartel,Renault S.A.S.\nhttps://competition-cases.ec.e...,01.04.2025,E.38.31 - Dismantling of wrecks (NACE Rev. 2),Art. 101 TFEU + Art. 53 EEA,NaN,01.04.2025 - Settlement Decision\n Decision t...,Press communication\n Commission carries out ...
2,AT.40401,Second-hand Rolling Stock,Cartel,České dráhy\nhttps://competition-cases.ec.euro...,23.10.2024,"H.49.10 - Passenger rail transport, interurban...",Art. 101 TFEU,NaN,23.10.2024 - Prohibition Decision\n Decision ...,Oral Hearing of 14.12.2022\n EN published on ...
3,AT.40795,Food delivery services,Cartel,Delivery Hero SE\nhttps://competition-cases.ec...,23.07.2024,G.47.91 - Retail sale via mail order houses or...,Art. 101 TFEU + Art. 53 EEA,NaN,23.07.2024 - Initiation of Proceedings\n Deci...,NaN
4,AT.40882,IFF - deletion of data,Cartel,NaN,24.06.2024,"C.20.41 - Manufacture of soap and detergents, ...",NaN,NaN,24.06.2024 - Decision imposing fines\n Decisi...,NaN
...,...,...,...,...,...,...,...,...,...,...
176,AT.37851,PO/ Carlsberg + Heineken,Cartel,Heineken \nhttps://competition-cases.ec.europa...,NaN,NaN,Art. 101 TFEU,NaN,NaN,Press communication\n Commission closes carte...
177,AT.37786,euro-zone exchange charges - Austria,Cartel,Verband Österreichischer Banken und Bankiers,NaN,K.64.1 - Monetary intermediation (NACE Rev. 2),Art. 101 TFEU,NaN,NaN,Press communication\n Commission suspects all...
178,AT.37152,PO/Plasterboard,Cartel,Lafarge SA\nhttps://competition-cases.ec.europ...,NaN,C.23.62 - Manufacture of plaster products for ...,Art. 101 TFEU,NaN,NaN,Press communication\n Antitrust: Commission w...
179,AT.36072,GFU - Norwegian Gas Negotiation Committee,Cartel,Marathon Petroleum Norge A/S\nhttps://competit...,NaN,D.35.2 - Manufacture of gas; distribution of g...,Art. 101 TFEU,NaN,NaN,Press communication\n Commission successfully...


In [20]:
# rename columns
df = df_raw.rename(columns={
    'Case number': 'cartel_id',
    'Case title': 'cartel_name',
    'Companies': 'firms_raw',
    'Economic activities': 'nace_raw'
})

In [ ]:
# clean firm names
# extract firms, remove urls and split names
def extract_firms(text):
    names = [line for line in text.split('\n') if not line.startswith('http')]
    return [name.strip() for name in names if name.strip()]

df['firms_raw'] = df['firms_raw'].astype(str).str.strip()
df['firms_list'] = df['firms_raw'].apply(extract_firms)

In [ ]:
# extract nace code, name, and revision
# define regex pattern with named groups
pattern = r'(?P<nace_code>[A-Z]\.\d+(?:\.\d+)*) - (?P<nace_name>.*?) \(NACE Rev\. (?P<nace_rev>\d+)\)'

# extract all matches 
matches = df['nace_raw'].str.extractall(pattern)

# group matches by original row index, aggregate matched groups into lists
agg = matches.groupby(level=0).agg(list)

# assign lists back to df columns
df['nace_code'] = agg['nace_code']
df['nace_name'] = agg['nace_name']
df['nace_rev'] = agg['nace_rev']

In [ ]:
# clean legal basis
def extract_articles(text):
    if pd.isna(text):
        return []
    return [article.strip() for article in text.split('+') if article.strip()]

# clean and extract article list
df['Legal basis'] = df['Legal basis'].astype(str).str.strip()
df['legal_basis'] = df['Legal basis'].apply(extract_articles)

In [24]:
cartel_cols = ['cartel_id', 'cartel_name', 'firms_list', 'nace_code', 
              'nace_name', 'nace_rev', 'legal_basis']
df_cartel = df[cartel_cols]
df_cartel

,cartel_id,cartel_name,firms_list,nace_code,nace_name,nace_rev,legal_basis
0,AT.40636,SNBB,"[C.H. Boehringer Sohn AG & Co. KG, Alkaloids o...",[C.21.20],[Manufacture of pharmaceutical preparations],[2],"[Art. 101 TFEU, Art. 53 EEA]"
1,AT.40669,End-of-life vehicle recycling,"[Renault S.A.S., Mercedes-Benz Group AG, Mitsu...",[E.38.31],[Dismantling of wrecks],[2],"[Art. 101 TFEU, Art. 53 EEA]"
2,AT.40401,Second-hand Rolling Stock,"[České dráhy, Österreichische Bundesbahnen]",[H.49.10],"[Passenger rail transport, interurban]",[2],[Art. 101 TFEU]
3,AT.40795,Food delivery services,"[Delivery Hero SE, Glovoapp23 SA]","[G.47.91, I.56.1]",[Retail sale via mail order houses or via Inte...,"[2, 2]","[Art. 101 TFEU, Art. 53 EEA]"
4,AT.40882,IFF - deletion of data,[nan],[C.20.41],"[Manufacture of soap and detergents, cleaning ...",[2],[nan]
...,...,...,...,...,...,...,...
176,AT.37851,PO/ Carlsberg + Heineken,"[Heineken, Carlsberg International]",NaN,NaN,NaN,[Art. 101 TFEU]
177,AT.37786,euro-zone exchange charges - Austria,[Verband Österreichischer Banken und Bankiers],[K.64.1],[Monetary intermediation],[2],[Art. 101 TFEU]
178,AT.37152,PO/Plasterboard,"[Lafarge SA, Gyproc Benelux N.V., BPB Plc, Geb...",[C.23.62],[Manufacture of plaster products for construct...,[2],[Art. 101 TFEU]
179,AT.36072,GFU - Norwegian Gas Negotiation Committee,"[Marathon Petroleum Norge A/S, Esso Norge AS, ...",[D.35.2],[Manufacture of gas; distribution of gaseous f...,[2],[Art. 101 TFEU]


In [25]:
df_cartel.to_excel('../data/interim/cartel_cleaned.xlsx', index=False)

In [26]:
firms_cols = ['cartel_id', 'cartel_name', 'firms_list']
df_firms = df[firms_cols]
df_firms = df[firms_cols].explode('firms_list').rename(columns={'firms_list': 'firm'})
df_firms

,cartel_id,cartel_name,firm
0,AT.40636,SNBB,C.H. Boehringer Sohn AG & Co. KG
0,AT.40636,SNBB,Alkaloids of Australia Pty. Limited
0,AT.40636,SNBB,Transo-Pharm Holding-AG
0,AT.40636,SNBB,Ipsen S.A.
0,AT.40636,SNBB,Alkaloids Corporation
...,...,...,...
180,AT.35691,Pre-insulated pipe cartel,Ke-Kelit Kunststoffwerk GmbH
180,AT.35691,Pre-insulated pipe cartel,Dansk Rorindustri a/s
180,AT.35691,Pre-insulated pipe cartel,ABB Asea Brown Boveri Ltd
180,AT.35691,Pre-insulated pipe cartel,Tarco Energi A/S


In [27]:
df_firms.to_excel('../data/interim/cartel_exploded.xlsx', index=False)